# 🥗 Food Detection Version 2 Training (221 Food Categories - YOLOv8)### South Indian & Global Food Inspection AI Pipeline[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Food_Detection/NoteBooks/Food%20Detection/Version2/food_detection_v2_training.ipynb)---### 🚀 End-to-End Workflow:1. **Enable GPU**: Click `Runtime` -> `Change runtime type` -> Select **T4 GPU** (or A100 / L4).2. **Run All Cells**: Clones repository, dynamically resolves paths for 221 classes, trains YOLOv8 on 11,050 pure food photos.3. **Analytical Plots**: Displays loss curves, mAP progression, and confusion matrices.4. **1-Click Download**: Automatically downloads `food_detection_v2_best.pt` and `food_detection_v2_best.onnx` to your computer.

## Step 1: Install Dependencies & Verify GPU AccelerationInstalls Ultralytics, PyYAML, OpenCV, and verifies CUDA acceleration.

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics torchvision opencv-python-headless pyyaml matplotlib seaborn pandas onnx

import torch, sys, ultralytics
print('=' * 60)
print('🖥️  SYSTEM & HARDWARE SPECIFICATIONS')
print('=' * 60)
print(f'Python Version   : {sys.version.split()[0]}')
print(f'PyTorch Version  : {torch.__version__}')
print(f'Ultralytics      : {ultralytics.__version__}')
print(f'CUDA Available   : {"✅ Yes" if torch.cuda.is_available() else "❌ No (Please switch Colab runtime to GPU!)"}')
if torch.cuda.is_available():
    print(f'GPU Device Name  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print('=' * 60)
ultralytics.checks()
# Mount Google Drive upfront so all checkpoints auto-save in real-time
if 'google.colab' in str(get_ipython()):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive connected! All training epochs will save directly to Drive.')
    except Exception as e:
        print(f'Drive note: {e}')


## Step 2: Fresh GitHub Repository CloneClones the repository directly into Colab's high-speed local NVMe storage (`/content/`).

In [ ]:
import os

REPO_URL = 'https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git'
REPO_NAME = 'SouthIndian_Food_Nutrition'

# In Google Colab, perform fresh clone
if 'google.colab' in str(get_ipython()):
    if not os.path.exists(REPO_NAME):
        print(f'🚀 Freshly cloning {REPO_NAME} from GitHub...')
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    print(f'✅ Active Working Directory: {os.getcwd()}')
else:
    print(f'Running in local workspace: {os.getcwd()}')

## Step 3: Dataset Configuration (Version 2 - 221 Classes)Configures dynamic absolute paths in `dataset.yaml` for Google Colab.

In [ ]:
import os, yaml

candidate_yamls = [
    'Food_Detection/DataSet/Version2 Used Data/dataset.yaml',
    os.path.join(os.getcwd(), 'Food_Detection', 'DataSet', 'Version2 Used Data', 'dataset.yaml'),
    '/content/SouthIndian_Food_Nutrition/Food_Detection/DataSet/Version2 Used Data/dataset.yaml'
]

dataset_yaml_path = None
for p in candidate_yamls:
    if os.path.exists(p):
        dataset_yaml_path = os.path.abspath(p)
        break

if dataset_yaml_path:
    dataset_dir = os.path.dirname(dataset_yaml_path)
    print(f'✅ Found Version 2 dataset.yaml at: {dataset_yaml_path}')
    print(f'📁 Dataset root directory: {dataset_dir}')
    
    with open(dataset_yaml_path, 'r', encoding='utf-8') as f:
        data_cfg = yaml.safe_load(f)
    
    data_cfg['path'] = dataset_dir
    data_cfg['train'] = 'images/train'
    data_cfg['val'] = 'images/val'
    data_cfg['test'] = 'images/test'
    
    with open(dataset_yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_cfg, f, sort_keys=False)
    
    names = data_cfg.get('names', {})
    print(f'\n🔥 Total Configured Classes (nc): {len(names)}')
    print('Sample Classes across Categories:')
    for s_id in [0, 2, 21, 56, 78, 116, 143, 162, 194, 220]:
        if s_id in names:
            print(f'  [{s_id:3d}] {names[s_id]}')
else:
    raise FileNotFoundError('❌ dataset.yaml not found! Please check repository path.')

## Step 4: Sample Visual InspectionVisual sanity check to display training samples with ground-truth food bounding boxes.

In [ ]:
import glob, cv2, matplotlib.pyplot as plt

train_img_dir = os.path.join(dataset_dir, 'images', 'train')
train_lbl_dir = os.path.join(dataset_dir, 'labels', 'train')
img_files = glob.glob(os.path.join(train_img_dir, '*.jpg'))[:4]

if img_files:
    fig, axes = plt.subplots(1, len(img_files), figsize=(20, 5))
    if len(img_files) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, img_files):
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(train_lbl_dir, f'{stem}.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:5])
                        x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                        x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cname = names.get(cid, str(cid))
                        cv2.putText(img, cname, (x1, max(y1 - 6, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(stem[:24], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No training samples found to preview.')

## Step 5: Launch YOLOv8 Model Training (221 Food Categories)Trains YOLOv8 on all 221 classes with automated checkpointing and metric tracking.

In [ ]:
from ultralytics import YOLO
import os

# Hyperparameters for 221 Classes
EPOCHS = 100         # 100 epochs for high mAP accuracy across 221 categories
BATCH_SIZE = 16      # 16 on T4 GPU
IMAGE_SIZE = 640
MODEL_BASE = 'yolov8s.pt'
RUN_NAME = 'food_detection_v2_run'

# If Google Drive is mounted, save directly into Google Drive in real-time!
if os.path.exists('/content/drive/MyDrive'):
    PROJECT_DIR = '/content/drive/MyDrive/Food_Inspection_V2_Models/Food_Detection'
    print(f'💾 Project Directory (Google Drive):\n   {PROJECT_DIR}/{RUN_NAME}')
else:
    PROJECT_DIR = 'Food_Detection/runs/train'
    print(f'📁 Project Directory (Local): {PROJECT_DIR}/{RUN_NAME}')

os.makedirs(PROJECT_DIR, exist_ok=True)

last_ckpt = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'last.pt')

# Check if previous training was interrupted to seamlessly resume!
if os.path.exists(last_ckpt):
    print('=' * 65)
    print(f'🔄 DETECTED SAVED CHECKPOINT AT: {last_ckpt}')
    print('🚀 RESUMING TRAINING SEAMLESSLY (Continuing remaining epochs)...')
    print('=' * 65)
    model = YOLO(last_ckpt)
    results = model.train(resume=True)
else:
    print('=' * 65)
    print('🚀 LAUNCHING FRESH YOLOV8 TRAINING (100 EPOCHS)')
    print('=' * 65)
    print(f'Model Base       : {MODEL_BASE}')
    print(f'Total Classes    : {len(names)}')
    print(f'Target Epochs    : {EPOCHS} (Early Stopping patience=15)')
    print(f'Batch Size       : {BATCH_SIZE}')
    print(f'Image Resolution : {IMAGE_SIZE}x{IMAGE_SIZE}')
    print('=' * 65)
    model = YOLO(MODEL_BASE)
    results = model.train(
        data=dataset_yaml_path,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=PROJECT_DIR,
        name=RUN_NAME,
        patience=15,       # Automatically stops if validation mAP peaks early
        save=True,
        save_period=5,     # Checkpoint saved every 5 epochs directly to Drive
        exist_ok=True,
        plots=True,
        verbose=True
    )

print('\n✅ Model Training Completed! Checkpoint is safe in Google Drive.')


## Step 6: Analytical Plots & Metric VisualizationDisplays the loss curves, mAP progression, and confusion matrix.

In [ ]:
import os, glob, matplotlib.pyplot as plt, matplotlib.image as mpimg

run_dir = os.path.join(PROJECT_DIR, RUN_NAME)
print('=' * 65)
print('📊 TRAINING ANALYTICS & PERFORMANCE PLOTS')
print('=' * 65)

# Primary Metrics Plot: Loss Curves & mAP Progression
results_plot = os.path.join(run_dir, 'results.png')
if os.path.exists(results_plot):
    print('📈 1. Training & Validation Curves (Loss, Precision, Recall, mAP50, mAP50-95):')
    plt.figure(figsize=(18, 9))
    plt.imshow(mpimg.imread(results_plot))
    plt.title('YOLOv8 Version 2 - Training Curves (results.png)', fontsize=15, pad=12)
    plt.axis('off')
    plt.show()

# Confusion Matrix
cm_plot = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(cm_plot):
    print('\n🎯 2. Confusion Matrix:')
    plt.figure(figsize=(16, 16))
    plt.imshow(mpimg.imread(cm_plot))
    plt.title('YOLOv8 Version 2 - Confusion Matrix', fontsize=15, pad=12)
    plt.axis('off')
    plt.show()

# Validation Batch Predictions vs Ground Truth
val_preds = glob.glob(os.path.join(run_dir, 'val_batch*_pred.jpg'))[:2]
for vp in val_preds:
    print(f'\n🖼️ 3. Validation Predictions: {os.path.basename(vp)}')
    plt.figure(figsize=(16, 12))
    plt.imshow(mpimg.imread(vp))
    plt.title(f'Predictions Sample: {os.path.basename(vp)}', fontsize=14, pad=10)
    plt.axis('off')
    plt.show()

## Step 7: Standardize Weights & Export to ONNXSaves standardized model artifacts with exact names: `food_detection_v2_best.pt` and `food_detection_v2_best.onnx`.

In [ ]:
import shutil

weights_dir = os.path.join(run_dir, 'weights')
best_pt_src = os.path.join(weights_dir, 'best.pt')
last_pt_src = os.path.join(weights_dir, 'last.pt')

OUTPUT_MODELS_DIR = 'Food_Detection/Models'
os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)

STANDARDIZED_BEST_PT = os.path.join(OUTPUT_MODELS_DIR, 'food_detection_v2_best.pt')
STANDARDIZED_LAST_PT = os.path.join(OUTPUT_MODELS_DIR, 'food_detection_v2_last.pt')
STANDARDIZED_BEST_ONNX = os.path.join(OUTPUT_MODELS_DIR, 'food_detection_v2_best.onnx')

if os.path.exists(best_pt_src):
    shutil.copy(best_pt_src, STANDARDIZED_BEST_PT)
    shutil.copy(best_pt_src, 'food_detection_v2_best.pt')  # copy to current dir for easy 1-click download
    print(f'🏆 Best Model Saved: {STANDARDIZED_BEST_PT}')

if os.path.exists(last_pt_src):
    shutil.copy(last_pt_src, STANDARDIZED_LAST_PT)
    print(f'📦 Last Model Saved: {STANDARDIZED_LAST_PT}')

# Export to ONNX
try:
    print('\n⚡ Exporting Best Model to ONNX format...')
    best_model = YOLO(STANDARDIZED_BEST_PT)
    onnx_file = best_model.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True)
    if os.path.exists(onnx_file):
        shutil.copy(onnx_file, STANDARDIZED_BEST_ONNX)
        shutil.copy(onnx_file, 'food_detection_v2_best.onnx')
        print(f'⚡ ONNX Export Succeeded: {STANDARDIZED_BEST_ONNX}')
except Exception as e:
    print(f'ONNX Export Note: {e}')

## Step 8: 📥 1-Click Automatic Model DownloadAutomatically downloads `food_detection_v2_best.pt` and `food_detection_v2_best.onnx` to your computer.

In [ ]:
# =====================================================================
# 💾 STEP 8: AUTOMATED OVERNIGHT PERSISTENCE & 1-CLICK AUTO-DOWNLOAD
# =====================================================================
# If training takes hours and you leave/go home:
# 1. Permanently saves models to your personal Google Drive (survives disconnects & sleep)
# 2. Triggers automatic local browser download

import os, shutil

MODEL_FILES = ['food_detection_v2_best.pt', 'food_detection_v2_best.onnx', 'results.png']

# --- 1. PERMANENT GOOGLE DRIVE BACKUP (SAFE OVERNIGHT STORAGE) ---
try:
    from google.colab import drive
    print('=' * 65)
    print('💾 SAVING TO GOOGLE DRIVE FOR PERMANENT OVERNIGHT SAFETY')
    print('=' * 65)
    drive.mount('/content/drive', force_remount=False)
    
    gdrive_dir = '/content/drive/MyDrive/Food_Inspection_V2_Models/Food_Detection'
    os.makedirs(gdrive_dir, exist_ok=True)
    
    for f in MODEL_FILES:
        if os.path.exists(f):
            shutil.copy2(f, os.path.join(gdrive_dir, f))
            print(f'✅ Permanently backed up to Google Drive: {gdrive_dir}/{f}')
            
    print('\n🎉 SUCCESS: All models are permanently stored in your Google Drive!')
    print('Even if you close your laptop, turn off your PC, or go home, your model is 100% safe.')
except Exception as e:
    print(f'Google Drive sync note: {e}')

# --- 2. AUTOMATIC BROWSER DOWNLOAD ---
try:
    from google.colab import files
    print('\n' + '=' * 65)
    print('📥 TRIGGERING AUTOMATIC DIRECT FILE DOWNLOAD')
    print('=' * 65)
    for f in ['food_detection_v2_best.pt', 'food_detection_v2_best.onnx']:
        if os.path.exists(f):
            print(f'Downloading {f} to your computer...')
            files.download(f)
except ImportError:
    print('Running locally: Weights are stored in Food_Detection/Models/')